In [ ]:
import pandas as pd
import copy
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from pmdarima import auto_arima
from prophet import Prophet
import optuna


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def format_as_year_month(df):
    df['ds'] = df['year'].astype(str) + '-' + df['month'].astype(str)
    df.index = pd.to_datetime(df['ds'])
    df.drop(['ds'], axis = 1, inplace = True)
    df = df.asfreq('MS')
    return df   

def detect_outlier(ds, threshold = 3):
    mean_1    = np.mean(ds)
    std_1     = np.std(ds)
    z_score  = (ds - mean_1)/std_1
    outliers = ds[abs(z_score)>threshold]
    return outliers

def limpiar_outliers_x_agrupacion(df_real, agrupacion, target_var):
    df = df_real.copy()
    n = 0
    idx_outliers_total =  []
    for group in df[agrupacion].unique():
        n += 1
        df_group = df[df[agrupacion]==group]
        idx_outliers = list(detect_outlier(df_group[target_var], 3.5).index)
        if len(idx_outliers) >0:
            promedios    = df_group[~df_group.index.isin(idx_outliers)].mean()
            df.loc[df.index.isin(idx_outliers), target_var] = promedios[target_var]
            idx_outliers_total = idx_outliers_total + idx_outliers
    return df, idx_outliers_total

def feature_selection(df_target, exog, corte, max_lag, min_lag, target_col, ignore_col, negatives_reg_col):
    selection = []
    df = df_target.drop(['year', 'month'], axis=1).join(exog, how='inner')
    df_corr_lag = {i: pd.DataFrame() for i in range(min_lag, max_lag + 1)}
    variables = [x for x in df.columns if x not in [target_col] + ignore_col]
    for var in variables:
        print(target_col)
        corr = sm.tsa.stattools.ccf(df[target_col], df[var], adjusted=False)
        if var in negatives_reg_col:
            for lag in range(min_lag, max_lag + 1):
                if lag >= 0: 
                    corr_value = corr[lag]
                else: 
                    corr_value = sm.tsa.stattools.ccf( df[target_col], df[var],adjusted=False)[abs(lag)]
                df_corr_lag[lag] = df_corr_lag[lag].append({'feature': var, 'corr': corr_value}, ignore_index=True)
        else:
            for lag in range(0, max_lag + 1):
                if lag >= 0: 
                    corr_value = corr[lag]
                else: 
                    corr_value = sm.tsa.stattools.ccf( df[target_col], df[var],adjusted=False)[abs(lag)]
                df_corr_lag[lag] = df_corr_lag[lag].append({'feature': var, 'corr': corr_value}, ignore_index=True)
    for lag in range(min_lag, max_lag + 1):
            selected = df_corr_lag[lag][abs(df_corr_lag[lag]['corr']) >= corte]
            for _, row in selected.iterrows():
                selection.append({'lag': lag, 'variable': row['feature'], 'correlación': row['corr']})
    return selection

def feature_selection(df_target, exog, max_lag, min_lag, target_col, ignore_col, negatives_reg_col):
    results = []
    df = df_target.drop(['year', 'month'], axis=1).join(exog, how='inner')
    variables = [x for x in df.columns if x not in [target_col] + ignore_col]
    for var in variables:
        corr = sm.tsa.stattools.ccf(df[target_col], df[var], adjusted=False)
        if var in negatives_reg_col:
            lags = range(min_lag, max_lag + 1)
        else:
            lags = range(0, max_lag + 1)
        for lag in lags:
            lag_index = abs(lag)
            if lag_index < len(corr): 
                corr_value = corr[lag_index]
                results.append({
                    'variable': var,
                    'lag': lag,
                    'correlación': corr_value
                })
    return pd.DataFrame(results)

def clean_focus_correlation(df,group,focus): 
    return(df.loc[df.groupby(group)[focus].idxmax()].reset_index(drop=True))

def collinearity_analysis(df_target, exog, feat_select, target_col, corte_correl):
    df = df_target.drop(['year', 'month'], axis=1).join(exog, how='inner')
    df_lags = pd.DataFrame()
    for i, row in feat_select.iterrows():
        var = row['variable']
        lag = row['lag']
        df_lags[f'{var}_lag{lag}'] = df[var].shift(lag)
    df_lags = df_lags.dropna()
    correl_matrix = df_lags.corr()
    seleccionadas = []
    grupos_colineales = []
    ya_revisadas = set() 
    for var in correl_matrix.columns:
        if var not in ya_revisadas:
            grupo = correl_matrix[abs(correl_matrix[var]) > corte_correl].index.tolist()
            ya_revisadas.update(grupo)
            grupos_colineales.append(grupo)
    df_lags=pd.concat([df_lags, df['sale_amount_MM']],axis=1).dropna(axis=0)
    mape_grupos = []
    RESUMEN = []
    seleccionadas = []
    usadas = set()  
    for grupo in grupos_colineales:
        grupo_filtrado = [var for var in grupo if var not in usadas]
        if not grupo_filtrado:
            continue  
        correl_con_target = df_lags[[f'{target_col}'] + grupo_filtrado].corr()[target_col].drop(target_col)
        mejor_variable = correl_con_target.idxmax()
        seleccionadas.append(mejor_variable)
        usadas.add(mejor_variable)  
        mape_grupos.append(correl_con_target)
        RESUMEN.append([
            mejor_variable.split('_lag')[0],
            mejor_variable,
            np.round(list(correl_con_target.values), 2),
            grupo_filtrado
        ])
    return([eliminar_sufixo(col) for col in seleccionadas],RESUMEN)


def eliminar_sufixo(texto):
    return texto.split('_lag')[0]

In [ ]:
df = pd.read_pickle("data_family.pkl")
exog=df['exog']
sales_by_family=df['sales_by_family']
familias = pd.read_csv("familias.csv")
data_by_family = {}
for family in sales_by_family['family'].unique():
    aux = sales_by_family[sales_by_family['family'] == family].copy()
    aux = format_as_year_month(aux)
    data_by_family[family] = aux

data_by_family_imputado =  copy.deepcopy(data_by_family)
for family in data_by_family.keys():
    d = data_by_family_imputado[family]
    d_imputado, idx_outliers = limpiar_outliers_x_agrupacion(d, 'family', 'sale_amount_MM')
    data_by_family_imputado[family].loc[:,'sale_amount_MM'] = d_imputado['sale_amount_MM']
exog.ffill(inplace=True)
exog['Verano']= exog['Verano'].astype(float)
name='0101'
feat_select=feature_selection(data_by_family[name],exog,max_lag=4,min_lag=-3,target_col='sale_amount_MM',
                                ignore_col=['units_sold', 'avg_price', 'family', 'total_UCI',
                                            'retiro 1', 'retiro 2', 'retiro 3','feriados','dias_del_mes'],
                                negatives_reg_col=[
                                            'Primavera','Verano','Otono', 'Halloween','Navidad','Fiestas_patrias','Dia_mama',
                                            'san_valentin','Norte_tavg','Norte_prcp','Norte_wspd','Centro_tavg','Centro_prcp',
                                            'Centro_wspd','Sur_tavg','Sur_prcp','Sur_wspd','ExtremoSur_tavg','ExtremoSur_prcp',
                                                'ExtremoSur_wspd', 'cyber','cyber_pct_mes'] )
feat_select=feat_select[feat_select.correlación>0.6]
feat_select=pd.DataFrame(feat_select).reset_index(drop=True)
feat_select=clean_focus_correlation(pd.DataFrame(feat_select),group='variable',focus='correlación')
feat_select=feat_select.reindex(feat_select['correlación'].abs().sort_values(ascending=False).index)
feat_select=feat_select.reset_index().drop(columns=['index'], errors='ignore')
selected_colinealidad=collinearity_analysis(data_by_family[name], exog, feat_select, 'sale_amount_MM',0.7)
feat_select=feat_select.iloc[np.where(np.isin(feat_select.variable,selected_colinealidad[0]))]
target_col='sale_amount_MM'
data_selection=data_by_family[name].copy()
data_selection['ds'] = data_selection.index
data_selection['y'] = data_selection[target_col]
top_features = feat_select.copy()
lagged_df = pd.DataFrame(index=exog.index)
family=data_by_family[name]
family['y'] = family[target_col]
df = family.drop(['year', 'month'], axis=1).join(exog[list(feat_select.variable)].copy(), how='inner')
df.index = pd.to_datetime(df.index)
df=df.loc['2017-01-01':'2024-12-01']
df_regressors = pd.DataFrame()
for index, row in feat_select.iterrows():
        df_regressors[row['variable'] +'_lag_' + str(row['lag'])] = df[row['variable']].shift(row['lag'])
df =df[['y']].join(df_regressors.dropna())
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('/', '_')
df.columns = df.columns.str.replace('-', '_')
df = df.dropna()
df = df.rename(columns={'index': 'ds'})
np.random.seed(12)
test_periods = 12  
train_periods = 72 
split = []
split=[(list(range(i, i + train_periods)),list(range(i + train_periods, i + train_periods + test_periods))) for i in range(0,19) if (i + train_periods + test_periods - 1) < len(df)]


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:1179: RuntimeWarning: invalid value encountered in divide
  ret = cvf / (np.std(x) * np.std(y))


In [5]:

def fit_predict_eval_hw(training_set, test_set, model_params = None):
    scaler=None
    if model_params is None:
        trend = 'add'
        seasonal = 'add'
        use_boxcox = False
        seasonal_periods = 12
    else:
        trend = model_params['trend']
        seasonal = model_params['seasonal']
        use_boxcox = False
        seasonal_periods = model_params['seasonal_periods']
    model= ExponentialSmoothing(endog=training_set['y'].dropna(), 
                                initialization_method='estimated',
                                freq='MS',
                                seasonal_periods = seasonal_periods, 
                                trend = trend,
                                seasonal = seasonal,
                                use_boxcox = use_boxcox )
    ets=model.fit()
    y_pred = pd.Series(ets.predict(start=test_set.index[0],end=test_set.index[-1])).rename('ETS')
    return ets, y_pred,scaler

def optimize_hw_cv(df, split, n_trials=50):
    model_dict = {}
    def objective(trial):
        trend = trial.suggest_categorical('trend', ['add', 'mul', None])
        seasonal = trial.suggest_categorical('seasonal', ['add', 'mul', None])
        seasonal_periods = trial.suggest_int('seasonal_periods', 2, 24)
        model_params = {
            'trend': trend,
            'seasonal': seasonal,
            'seasonal_periods': seasonal_periods
        }
        mape_scores = []  # Lista para almacenar MAPE por fold
        best_mape = np.inf  # Mejor MAPE de un fold específico
        best_model = None  # Mejor modelo entrenado
        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index].fillna(0)
                test_set = df.iloc[test_index].fillna(0)
                model, y_pred,_ = fit_predict_eval_hw(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)
                if mape < best_mape:
                    best_mape = mape
                    best_model = model
            trial_number = trial.number
            model_dict[trial_number] = {
                'model': best_model,
                'params': model_params,
                'mape_best': best_mape
            }
            return np.mean(mape_scores)
        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value  # MAPE promedio de todos los folds
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, mape_best))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best'])
    best_params = study.best_params
    return best_params, trials_df, model_dict

def fit_predict_eval_sarimax(training_set, test_set, model_params = None):
    scaler=None
    if model_params is None:
        p = 1
        d = 0
        q = 1
        P = 1
        D = 0
        Q = 1
        s = 12
    else:
        p = model_params['p']
        d = model_params['d']
        q = model_params['q']
        P = model_params['P']
        D = model_params['D']
        Q = model_params['Q']
        s = model_params['s']
    model=sm.tsa.SARIMAX(endog=training_set['y'], exog=training_set.drop(['y','ds'], axis = 1), order=(p,d,q), seasonal_order = (P,D,Q,s))
    arima=model.fit(disp=False,maxiter=1000)
    y_pred = pd.Series(arima.predict(start=test_set.index[0],end=test_set.index[-1],dynamic=True,exog=test_set.drop(['y', 'ds'], axis = 1))).rename('arima')
    return arima, y_pred,scaler


def optimize_sarimax_cv(df, split, n_trials=50):
    model_dict = {}
    def objective(trial):
        p = trial.suggest_int('p', 0, 3)
        d = trial.suggest_int('d', 0, 2)
        q = trial.suggest_int('q', 0, 3)
        P = trial.suggest_int('P', 0, 2)
        D = trial.suggest_int('D', 0, 1)
        Q = trial.suggest_int('Q', 0, 2)
        s = trial.suggest_int('s', 2, 24)  # Período estacional
        model_params = {
            'p': p, 'd': d, 'q': q,
            'P': P, 'D': D, 'Q': Q, 's': s
        }
        mape_scores = []  # Lista para almacenar MAPE por fold
        best_mape = np.inf  # Mejor MAPE de un fold
        best_model = None  # Mejor modelo entrenado
        try:
            for train_index, test_index in split:
                # Separar datos en entrenamiento y prueba
                training_set = df.iloc[train_index].fillna(0)
                test_set = df.iloc[test_index].fillna(0)
                # Entrenar y predecir con SARIMAX
                model, y_pred,_  = fit_predict_eval_sarimax(training_set, test_set, model_params)
                # Calcular MAPE
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)
                # Verificar si este fold tiene el menor MAPE
                if mape < best_mape:
                    best_mape = mape
                    best_model = model
            # Guardar el mejor modelo para este ensayo
            trial_number = trial.number
            model_dict[trial_number] = {
                'model': best_model,
                'params': model_params,
                'mape_best': best_mape
            }

            # Retornar el MAPE promedio como objetivo
            return np.mean(mape_scores)
        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf 
    # Configuración del estudio de Optuna
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    # Crear el DataFrame con resultados de los ensayos
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value  # MAPE promedio de todos los folds
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, mape_best))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )
    best_params = study.best_params
    return best_params, trials_df, model_dict


def fit_predict_eval_mlp(train_data, test_data, model_params):
    date_col='ds'
    target_col='y'
    if model_params is None:
        hidden_layer_sizes = (64, 32)
        max_iter =500
    else:
        hidden_layer_sizes =model_params['hidden_layer_sizes']
        max_iter =model_params['max_iter']
    X_train = train_data.drop(columns=[date_col, target_col])
    y_train = train_data[target_col].values
    X_test = test_data.drop(columns=[date_col, target_col])
    y_test = test_data[target_col].values
    scaler_X = MinMaxScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    model = MLPRegressor(hidden_layer_sizes=hidden_layer_sizes, max_iter=max_iter, random_state=0)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    forecast_test_df = pd.DataFrame({
        date_col: test_data[date_col].values,
        'pred_test': y_pred})
    return model,forecast_test_df,scaler_X


def optimize_mlp_cv(df, split, n_trials=50):
    model_dict = {}

    def objective(trial):
        # Definir los hiperparámetros a optimizar
        hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(64, 32), (128, 64), (256, 128)])
        max_iter = trial.suggest_int('max_iter', 200, 1000, step=100)

        model_params = {
            'hidden_layer_sizes': hidden_layer_sizes,
            'max_iter': max_iter
        }
        mape_scores = []  # MAPE para cada fold
        best_mape = np.inf  # Mejor MAPE entre los folds
        best_model_params = None  # Parámetros del mejor modelo entre los folds
        best_model = None  # Mejor modelo entrenado
        for train_indices, test_indices in split:
            train_data = df.iloc[train_indices].copy()
            test_data = df.iloc[test_indices].copy()
            model, forecast_test_df,_  = fit_predict_eval_mlp(train_data, test_data, model_params)
            y_true = test_data['y'].values
            y_pred = forecast_test_df['pred_test'].values
            mape = mean_absolute_percentage_error(y_true, y_pred)
            mape_scores.append(mape)
            if mape < best_mape:
                best_mape = mape
                best_model_params = model_params
                best_model = model
        trial_number = trial.number
        model_dict[trial_number] = {
            'model': best_model,
            'params': best_model_params,
            'mape_best': best_mape
        }
        return np.mean(mape_scores)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        best_mape = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, best_mape))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )
    best_params = study.best_params
    return best_params, trials_df, model_dict

from sklearn.linear_model import ElasticNet

def fit_predict_eval_elastic_net(train_data, test_data, model_params):
    date_col = 'ds'
    target_col = 'y'
    if model_params is None:
        alpha = 1.0
        l1_ratio = 0.5
    else:
        alpha = model_params.get('alpha', 1.0)
        l1_ratio = model_params.get('l1_ratio', 0.5)
    X_train = train_data.drop(columns=[date_col, target_col])
    y_train = train_data[target_col].values
    X_test = test_data.drop(columns=[date_col, target_col])
    y_test = test_data[target_col].values
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=0)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    forecast_test_df = pd.DataFrame({
        date_col: test_data[date_col].values,
        'pred_test': y_pred})
    return model, forecast_test_df,scaler



def optimize_elastic_net_cv(df, split, n_trials=50):
    model_dict = {}
    def objective(trial):
        # Definir los hiperparámetros a optimizar
        alpha = trial.suggest_loguniform('alpha', 1e-5, 1e1)
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        model_params = {'alpha': alpha, 'l1_ratio': l1_ratio}
        mape_scores = []  # MAPE para cada fold
        best_mape = np.inf  # Mejor MAPE entre los folds
        best_model_params = None  # Parámetros del mejor modelo entre los folds
        best_model = None  # Mejor modelo entrenado

        for train_indices, test_indices in split:
            train_data = df.iloc[train_indices].copy()
            test_data = df.iloc[test_indices].copy()
            model, forecast_test_df,_  = fit_predict_eval_elastic_net(train_data, test_data, model_params)

            y_true = test_data['y'].values
            y_pred = forecast_test_df['pred_test'].values
            mape = mean_absolute_percentage_error(y_true, y_pred)
            mape_scores.append(mape)
            if mape < best_mape:
                best_mape = mape
                best_model_params = model_params
                best_model = model
        trial_number = trial.number
        model_dict[trial_number] = {
            'model': best_model,
            'params': best_model_params,
            'mape_best': best_mape
        }
        return np.mean(mape_scores)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        best_mape = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, best_mape))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )
    best_params = study.best_params
    return best_params, trials_df, model_dict




def fit_predict_eval_prophet(training_set, test_set, model_params = None):
    scaler=None
    if model_params is None:
        m = Prophet(
            growth = 'linear',
            seasonality_prior_scale=0.01,
            seasonality_mode = "multiplicative",
            holidays_prior_scale=5,
            changepoint_prior_scale=0.01,
            n_changepoints=25)
    else:
         m = Prophet(

            growth = model_params['growth'],
            seasonality_prior_scale = model_params['seasonality_prior_scale'],
            seasonality_mode = model_params['seasonality_mode'],
            holidays_prior_scale = model_params['holidays_prior_scale'],
            changepoint_prior_scale = model_params['changepoint_prior_scale'],
            n_changepoints = model_params['n_changepoints'] )

    m.add_country_holidays(country_name='CL')
    for col in training_set.drop(['y','ds'], axis = 1).columns:
        m.add_regressor(col)
    m.fit(training_set)
    future = m.make_future_dataframe(periods = test_set.shape[0], freq = 'MS')
    future = future.merge(pd.concat([training_set, test_set]), on='ds', how='left')
    predictions_prophet = m.predict(future.dropna())[-test_set.shape[0]:]['yhat'].rename('Prophet')
    return m, predictions_prophet,scaler


 
def optimize_prophet_cv(df, split, n_trials=10):
    model_dict = {}
    def objective(trial):
        # Definir los hiperparámetros
        growth = trial.suggest_categorical('growth', ['linear', 'flat'])
        seasonality_prior_scale = trial.suggest_float('seasonality_prior_scale', 0.01, 30, log=True)
        seasonality_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
        holidays_prior_scale = trial.suggest_float('holidays_prior_scale', 5, 20, log=True)
        changepoint_prior_scale = trial.suggest_float('changepoint_prior_scale', 0.01, 2)
        n_changepoints = trial.suggest_int('n_changepoints', 25, 50)
        model_params = {
            'growth': growth,
            'seasonality_prior_scale': seasonality_prior_scale,
            'seasonality_mode': seasonality_mode,
            'holidays_prior_scale': holidays_prior_scale,
            'changepoint_prior_scale': changepoint_prior_scale,
            'n_changepoints': n_changepoints
        }
        mape_scores = []  # MAPE para cada fold
        best_mape = np.inf  # Mejor MAPE dentro de la validación cruzada
        best_model_params = None  # Parámetros del mejor modelo en validación cruzada
        try:
            for train_index, test_index in split:
                # Dividir datos
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]
                # Entrenar y predecir con Prophet
                _, y_pred,_  = fit_predict_eval_prophet(training_set, test_set, model_params)
                # Calcular MAPE
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)
                # Actualizar el mejor modelo en validación cruzada
                if mape < best_mape:
                    best_mape = mape
                    best_model_params = model_params
            # Guardar resultados en el diccionario de modelos
            trial_number = trial.number
            model_dict[trial_number] = {
                'params': model_params,
                'mape_best': best_mape
            }
            # Retornar el MAPE promedio de la validación cruzada como objetivo
            return np.mean(mape_scores)
        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf
    # Crear y optimizar el estudio de Optuna
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    # Crear el DataFrame con resultados de pruebas
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        best_model_params = model_dict.get(trial_number, {}).get('params', {})
        trials_data.append((trial_number, params, mape_mean, mape_best))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )
    # Retornar los mejores parámetros, resultados y modelos
    best_params = study.best_params
    return best_params, trials_df, model_dict

In [ ]:
best_params_hw, trials_df_hw,model_dict_hw=optimize_hw_cv(df, split, n_trials=10)
trials_df_hw['model']='holt_winters'
best_params_sarimax, trials_df_sarimax,model_dict_sarimax=optimize_sarimax_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_sarimax['model']='sarimax'
best_params_mlp, trials_df_mlp, model_dict_mlp=optimize_mlp_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_mlp['model']='mlp'
best_params_enet, trials_df_enet, model_dict_enet=optimize_elastic_net_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_enet['model']='elastic_net'
best_params_prophet, trials_df_prophet,model_dict_prophet=optimize_prophet_cv(df.reset_index(), split, n_trials=5)
trials_df_prophet['model']='prophet'
best_params_arima, trials_df_arima, model_dict_arima = optimize_autoarima_cv(df.reset_index(), split, n_trials=5)
trials_df_arima['model'] = 'autoarima'

In [6]:
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_percentage_error

import pandas as pd
import numpy as np

def fit_predict_eval_autoarima(training_set, test_set, model_params=None):
    if model_params is None:
        model_params = {
            'seasonal': True,
            'm': 12,
            'stepwise': True,
            'suppress_warnings': True,
            'error_action': 'ignore'
        }

    # Forzar m = 1 si no es estacional
    if not model_params.get('seasonal', True):
        model_params['m'] = 1

    # Validar y limpiar datos
    y_train = training_set['y'].dropna()
    y_test = test_set['y'].dropna()

    # Evitar entrenamiento con datos vacíos
    if len(y_train) == 0 or len(y_test) == 0:
        raise ValueError("Training or test set está vacío después de eliminar NaNs.")

    # Entrenar modelo
    model = auto_arima(y_train, **model_params)
    
    # Predecir
    y_pred = model.predict(n_periods=len(y_test))
    
    # Alinear índice si es necesario
    y_pred_series = pd.Series(y_pred, index=test_set.index[:len(y_pred)], name='AutoARIMA')
    
    return model, y_pred_series, None

import optuna
import numpy as np
import pandas as pd

def optimize_autoarima_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        seasonal = trial.suggest_categorical('seasonal', [True, False])
        m = trial.suggest_categorical('m', [1, 3, 6, 12])
        stepwise = trial.suggest_categorical('stepwise', [True, False])
        model_params = {
            'seasonal': seasonal,
            'm': m,
            'stepwise': stepwise,
            'suppress_warnings': True,
            'error_action': 'ignore'
        }

        mape_scores = []
        best_mape = np.inf
        best_model_params = None

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_autoarima(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape
                    best_model_params = model_params

            trial_number = trial.number
            model_dict[trial_number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    # Resultados de los trials
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        best_model_params = model_dict.get(trial_number, {}).get('params', {})
        trials_data.append((trial_number, params, mape_mean, mape_best))

    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    best_params = study.best_params
    return best_params, trials_df, model_dict

In [7]:
df = df[df['y'].notnull()].copy()
best_params_arima, trials_df_arima, model_dict_arima = optimize_autoarima_cv(df.reset_index(), split, n_trials=5)
trials_df_arima['model'] = 'autoarima'

[I 2025-05-04 01:09:39,229] A new study created in memory with name: no-name-1cf1b769-6bbf-47e5-909d-fb3ec4b5c74e
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
[I 2025-05-04 01:09:40,100] Trial 0 finished with value: inf and parameters: {'seasonal': False, 'm': 12, 'stepwise': False}. Best is trial 0 with value: inf.


Error en el ensayo con parámetros {'seasonal': False, 'm': 1, 'stepwise': False, 'suppress_warnings': True, 'error_action': 'ignore'}: Input contains NaN.


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
[I 2025-05-04 01:09:40,744] Trial 1 finished with value: inf and parameters: {'seasonal': False, 'm': 3, 'stepwise': True}. Best is trial 0 with value: inf.


Error en el ensayo con parámetros {'seasonal': False, 'm': 1, 'stepwise': True, 'suppress_warnings': True, 'error_action': 'ignore'}: Input contains NaN.


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
[I 2025-05-04 01:09:50,658] Trial 2 finished with value: inf and parameters: {'seasonal': True, 'm': 6, 'stepwise': True}. Best is trial 0 with value: inf.


Error en el ensayo con parámetros {'seasonal': True, 'm': 6, 'stepwise': True, 'suppress_warnings': True, 'error_action': 'ignore'}: Input contains NaN.


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
[I 2025-05-04 01:09:51,327] Trial 3 finished with value: inf and parameters: {'seasonal': False, 'm': 1, 'stepwise': True}. Best is trial 0 with value: inf.


Error en el ensayo con parámetros {'seasonal': False, 'm': 1, 'stepwise': True, 'suppress_warnings': True, 'error_action': 'ignore'}: Input contains NaN.


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
[I 2025-05-04 01:10:02,918] Trial 4 finished with value: inf and parameters: {'seasonal': True, 'm': 12, 'stepwise': True}. Best is trial 0 with value: inf.


Error en el ensayo con parámetros {'seasonal': True, 'm': 12, 'stepwise': True, 'suppress_warnings': True, 'error_action': 'ignore'}: Input contains NaN.


In [8]:
trials_df_arima

,trial_number,params,mape_mean,mape_best,model
0,0,"{'seasonal': False, 'm': 12, 'stepwise': False}",inf,inf,autoarima
1,1,"{'seasonal': False, 'm': 3, 'stepwise': True}",inf,inf,autoarima
2,2,"{'seasonal': True, 'm': 6, 'stepwise': True}",inf,inf,autoarima
3,3,"{'seasonal': False, 'm': 1, 'stepwise': True}",inf,inf,autoarima
4,4,"{'seasonal': True, 'm': 12, 'stepwise': True}",inf,inf,autoarima


In [9]:
pip install greykite

  Using cached holidays-0.13-py3-none-any.whl.metadata (11 kB)
Using cached holidays-0.13-py3-none-any.whl (172 kB)
  Attempting uninstall: holidays
    Found existing installation: holidays 0.71
    Uninstalling holidays-0.71:
      Successfully uninstalled holidays-0.71
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prophet 1.1.6 requires holidays<1,>=0.25, but you have holidays 0.13 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Greykite imports
from greykite.framework.templates.autogen.forecast_config import (
    ForecastConfig,
    MetadataParam,
    ModelComponentsParam,
    EvaluationPeriodParam
)
from greykite.framework.templates.forecaster import Forecaster
from greykite.framework.templates.model_templates import ModelTemplateEnum
from greykite.framework.utils.result_summary import summarize_grid_search_results
    
def fit_predict_eval_silverkite(training_set, test_set, model_params=None):
        scaler=None
        date_col='ds'
        target_col='y'
        if model_params is None:
                growth='linear'
                fit_algorithm='ridge'
                regularization_strength=0.6
        else:
                growth = model_params['growth']
                fit_algorithm = model_params['fit_algorithm']
                regularization_strength = model_params['regularization_strength']
            
        train_data=training_set.copy()
        test_data=test_set.copy()
        # Renombrar columnas para Silverkite
        train_data_silverkite = train_data.rename(columns={date_col: 'time', target_col: 'value'})
        train_data_silverkite['time'] = pd.to_datetime(train_data_silverkite['time'], errors='coerce')
        test_data_silverkite = test_data.rename(columns={date_col: 'time', target_col: 'value'})
        test_data_silverkite['time'] = pd.to_datetime(test_data_silverkite['time'], errors='coerce')
            

        date_col='time'
        target_col='value'
        metadata = MetadataParam(
            time_col='time',
            value_col='value',
            freq='MS',  # Frecuencia mensual
            train_end_date=train_data_silverkite['time'].iloc[-1]
        )

        # Configuración de regresores
        regressors = dict(
            regressor_cols=[col for col in train_data_silverkite.columns if col not in [date_col, target_col]]
        )

        # Configuración de eventos
        events = dict(
            holidays_to_model_separately="auto",
            holiday_lookup_countries=["CL"],
            holiday_pre_num_days=2,
            holiday_post_num_days=2,
            holiday_pre_post_num_dict=None,
        )

            # Configuración de changepoints
        changepoints = dict(
                changepoints_dict=dict(
                    method="custom",
                    regularization_strength=regularization_strength,
                    resample_freq="7D",
                    actual_changepoint_min_distance="100D",
                    potential_changepoint_distance="50D",
                    no_changepoint_proportion_from_end=0.3,
                    yearly_seasonality_order=6,
                    dates=["2020-05-23","2020-07-20","2020-12-10","2021-04-28"],  # Fechas personalizadas para puntos de cambio
                    combine_changepoint_min_distance="100D",
                    keep_detected=False,
                )
            )
            # Definir parámetros personalizados
        custom = dict(
                min_admissible_value=0,
                max_admissible_value=2e7,
                fit_algorithm_dict=dict(fit_algorithm=fit_algorithm)
            )



         # Configuración del modelo
        model_components = ModelComponentsParam(
                regressors=regressors,
                changepoints=changepoints,
                events=events,
                custom=custom,
                growth = dict(growth_term=growth)
            )

        # Configuración de predicción
        config = ForecastConfig(
                forecast_horizon=2+len(test_data_silverkite),  
                coverage=0.95, 
                metadata_param=metadata,
                model_components_param=model_components,
            )
        # Crear el objeto forecaster
        forecaster = Forecaster()

         # Entrenar el modelo y obtener los resultados de predicción
        gk_result = forecaster.run_forecast_config(
                df=pd.concat([train_data_silverkite, test_data_silverkite]),  
                config=config,
            )
        future_df = gk_result.timeseries.make_future_dataframe(
                periods=test_data_silverkite.shape[0],include_history=False)
        sk_fcst = gk_result.model.predict(future_df.merge(test_data_silverkite))
        predictions_sk = sk_fcst['forecast'].rename('Silverkite')
        return gk_result, predictions_sk,scaler


def optimize_silverkite_cv(df, split, n_trials=50):
    model_dict = {}
    def objective(trial):
        # Definir los hiperparámetros a optimizar
        growth = trial.suggest_categorical('growth', ['linear', 'quadratic', 'sqrt', None])
        fit_algorithm = trial.suggest_categorical('fit_algorithm', ['ridge', 'linear'])
        regularization_strength = trial.suggest_float('regularization_strength', 0.001, 10.0, log=True)
        model_params = {
            'growth': growth,
            'fit_algorithm': fit_algorithm,
            'regularization_strength': regularization_strength
        }
        mape_scores = []  # Lista para almacenar MAPE de cada fold
        best_mape = np.inf  # Mejor MAPE dentro de los folds
        best_model = None  # Mejor modelo entrenado dentro de los folds
        best_model_params = None  # Parámetros del mejor modelo
        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]
                model, y_pred,_  = fit_predict_eval_silverkite(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)
                if mape < best_mape:
                    best_mape = mape
                    best_model = model
                    best_model_params = model_params
            trial_number = trial.number
            model_dict[trial_number] = {
                'model': best_model,
                'params': best_model_params,
                'mape_best': best_mape
            }
            return np.mean(mape_scores)
        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value  # MAPE promedio de todos los folds
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        best_model_params = model_dict.get(trial_number, {}).get('params', {})
        trials_data.append((trial_number, params, mape_mean, mape_best))
    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )
    best_params = study.best_params
    return best_params, trials_df, model_dict



In [ ]:
best_params_silverkite, trials_df_silverkite,model_dict_silverkite=optimize_silverkite_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_silverkite['model']='silverkite'

In [10]:
trials_df_silverkite

,trial_number,params,mape_mean,mape_best,model
0,0,"{'growth': 'linear', 'fit_algorithm': 'linear'...",0.191437,0.114926,silverkite
1,1,"{'growth': 'sqrt', 'fit_algorithm': 'linear', ...",0.161101,0.097249,silverkite
2,2,"{'growth': 'linear', 'fit_algorithm': 'ridge',...",0.062093,0.050555,silverkite
3,3,"{'growth': 'linear', 'fit_algorithm': 'ridge',...",0.062093,0.050555,silverkite
4,4,"{'growth': 'quadratic', 'fit_algorithm': 'ridg...",0.063419,0.051869,silverkite
5,5,"{'growth': 'linear', 'fit_algorithm': 'linear'...",0.191437,0.114926,silverkite
6,6,"{'growth': 'sqrt', 'fit_algorithm': 'linear', ...",0.161101,0.097249,silverkite
7,7,"{'growth': 'linear', 'fit_algorithm': 'linear'...",0.191437,0.114926,silverkite
8,8,"{'growth': 'quadratic', 'fit_algorithm': 'ridg...",0.063419,0.051869,silverkite
9,9,"{'growth': 'quadratic', 'fit_algorithm': 'line...",0.300258,0.162251,silverkite


In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

def create_sequences(X, y, window_size):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])
    return np.array(X_seq), np.array(y_seq)

def fit_predict_eval_lstm(training_set, test_set, model_params=None):
    default_params = {
        'window_size': 12,
        'units': 50,
        'epochs': 50,
        'batch_size': 16,
        'learning_rate': 0.001
    }
    if model_params is not None:
        default_params.update(model_params)
    p = default_params

    features = training_set.drop(columns=['ds', 'y']).columns
    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_x.fit_transform(training_set[features])
    y_train = scaler_y.fit_transform(training_set[['y']])
    X_test = scaler_x.transform(test_set[features])
    y_test = scaler_y.transform(test_set[['y']])

    X_seq, y_seq = create_sequences(X_train, y_train, p['window_size'])

    model = Sequential()
    model.add(LSTM(p['units'], activation='tanh', input_shape=(p['window_size'], X_seq.shape[2])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=p['learning_rate']), loss='mse')
    model.fit(X_seq, y_seq, epochs=p['epochs'], batch_size=p['batch_size'], verbose=0)

    # Para predecir sobre test, usar últimas secuencias del train + test
    X_full = np.vstack([X_train, X_test])
    X_pred_seq = []
    for i in range(len(training_set), len(training_set) + len(test_set)):
        X_pred_seq.append(X_full[i - p['window_size']:i])
    X_pred_seq = np.array(X_pred_seq)

    y_pred_scaled = model.predict(X_pred_seq)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

    return model, pd.Series(y_pred, index=test_set.index, name='LSTM'), (scaler_x, scaler_y)

In [9]:
import optuna

def optimize_lstm_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            'window_size': trial.suggest_int('window_size', 6, 24),
            'units': trial.suggest_int('units', 16, 128),
            'epochs': trial.suggest_int('epochs', 20, 100),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_lstm(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape
            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }
            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, mape_best))

    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [13]:
best_params_lstm, trials_df_lstm, model_dict_lstm = optimize_lstm_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_lstm['model'] = 'lstm'

[I 2025-05-04 01:19:36,701] A new study created in memory with name: no-name-772daea9-a3d5-4195-98f3-e49bf1e6fc9e
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step


[I 2025-05-04 01:23:46,996] Trial 9 finished with value: 0.11041375003051125 and parameters: {'window_size': 17, 'units': 90, 'epochs': 20, 'batch_size': 32, 'learning_rate': 0.001133869144657798}. Best is trial 8 with value: 0.10480236218892112.


In [14]:
trials_df_lstm

,trial_number,params,mape_mean,mape_best,model
0,0,"{'window_size': 12, 'units': 101, 'epochs': 47...",0.105924,0.083014,lstm
1,1,"{'window_size': 19, 'units': 59, 'epochs': 51,...",0.112505,0.099281,lstm
2,2,"{'window_size': 8, 'units': 122, 'epochs': 67,...",0.118615,0.081915,lstm
3,3,"{'window_size': 23, 'units': 34, 'epochs': 72,...",0.121601,0.108568,lstm
4,4,"{'window_size': 19, 'units': 93, 'epochs': 33,...",0.119390,0.095756,lstm
5,5,"{'window_size': 24, 'units': 89, 'epochs': 86,...",0.117582,0.103845,lstm
6,6,"{'window_size': 23, 'units': 41, 'epochs': 88,...",0.113902,0.098862,lstm
7,7,"{'window_size': 21, 'units': 95, 'epochs': 96,...",0.126638,0.092192,lstm
8,8,"{'window_size': 9, 'units': 27, 'epochs': 68, ...",0.104802,0.090681,lstm
9,9,"{'window_size': 17, 'units': 90, 'epochs': 20,...",0.110414,0.084957,lstm


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np
import pandas as pd

def create_sequences(X, y, window_size):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])
    return np.array(X_seq), np.array(y_seq)

def fit_predict_eval_gru(training_set, test_set, model_params=None):
    default_params = {
        'window_size': 12,
        'units': 50,
        'epochs': 50,
        'batch_size': 16,
        'learning_rate': 0.001
    }
    if model_params:
        default_params.update(model_params)
    p = default_params

    features = training_set.drop(columns=['ds', 'y']).columns
    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_x.fit_transform(training_set[features])
    y_train = scaler_y.fit_transform(training_set[['y']])
    X_test = scaler_x.transform(test_set[features])
    y_test = scaler_y.transform(test_set[['y']])

    X_seq, y_seq = create_sequences(X_train, y_train, p['window_size'])

    model = Sequential()
    model.add(GRU(p['units'], activation='tanh', input_shape=(p['window_size'], X_seq.shape[2])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=p['learning_rate']), loss='mse')
    model.fit(X_seq, y_seq, epochs=p['epochs'], batch_size=p['batch_size'], verbose=0)

    # Armar secuencias de predicción sobre test
    X_full = np.vstack([X_train, X_test])
    X_pred_seq = []
    for i in range(len(training_set), len(training_set) + len(test_set)):
        X_pred_seq.append(X_full[i - p['window_size']:i])
    X_pred_seq = np.array(X_pred_seq)

    y_pred_scaled = model.predict(X_pred_seq)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

    return model, pd.Series(y_pred, index=test_set.index, name='GRU'), (scaler_x, scaler_y)

import optuna

def optimize_gru_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            'window_size': trial.suggest_int('window_size', 6, 24),
            'units': trial.suggest_int('units', 16, 128),
            'epochs': trial.suggest_int('epochs', 20, 100),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_gru(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, mape_best))

    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [16]:
best_params_gru, trials_df_gru, model_dict_gru = optimize_gru_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_gru['model'] = 'gru'

[I 2025-05-04 01:26:08,074] A new study created in memory with name: no-name-90c4f83d-bc7f-42cd-8952-05539ab88af5
/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


[I 2025-05-04 01:30:12,096] Trial 9 finished with value: 0.11037050732751469 and parameters: {'window_size': 19, 'units': 23, 'epochs': 72, 'batch_size': 8, 'learning_rate': 0.00016622071968705888}. Best is trial 8 with value: 0.09969500181351638.


In [17]:
trials_df_gru

,trial_number,params,mape_mean,mape_best,model
0,0,"{'window_size': 21, 'units': 98, 'epochs': 66,...",0.113111,0.097277,gru
1,1,"{'window_size': 13, 'units': 51, 'epochs': 83,...",0.102398,0.092497,gru
2,2,"{'window_size': 7, 'units': 57, 'epochs': 39, ...",0.107250,0.090828,gru
3,3,"{'window_size': 20, 'units': 19, 'epochs': 54,...",0.108867,0.092053,gru
4,4,"{'window_size': 8, 'units': 125, 'epochs': 56,...",0.104808,0.073945,gru
5,5,"{'window_size': 13, 'units': 62, 'epochs': 70,...",0.108027,0.084723,gru
6,6,"{'window_size': 14, 'units': 77, 'epochs': 24,...",0.110739,0.092119,gru
7,7,"{'window_size': 20, 'units': 93, 'epochs': 51,...",0.099939,0.063665,gru
8,8,"{'window_size': 12, 'units': 106, 'epochs': 87...",0.099695,0.083848,gru
9,9,"{'window_size': 19, 'units': 23, 'epochs': 72,...",0.110371,0.095606,gru


In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout
from tensorflow.keras.layers import MultiHeadAttention, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Sequential

def create_sequences(X, y, window_size):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:i+window_size])
        y_seq.append(y[i+window_size])
    return np.array(X_seq), np.array(y_seq)

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs  # ambas dimensiones coinciden

    # mantener misma dimensión para poder sumar después
    x = Dense(inputs.shape[-1])(res)  # proyecta de vuelta a input_dim
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)

    return x + res

def fit_predict_eval_transformer(training_set, test_set, model_params=None):
    p = {
        'window_size': 12,
        'head_size': 32,
        'num_heads': 2,
        'ff_dim': 64,
        'dropout': 0.1,
        'epochs': 50,
        'batch_size': 16,
        'learning_rate': 0.001
    }
    if model_params:
        p.update(model_params)

    features = training_set.drop(columns=['ds', 'y']).columns
    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_x.fit_transform(training_set[features])
    y_train = scaler_y.fit_transform(training_set[['y']])
    X_test = scaler_x.transform(test_set[features])
    y_test = scaler_y.transform(test_set[['y']])

    X_seq, y_seq = create_sequences(X_train, y_train, p['window_size'])

    inp = Input(shape=(p['window_size'], X_seq.shape[2]))
    x = transformer_encoder(inp, p['head_size'], p['num_heads'], p['ff_dim'], p['dropout'])
    x = Dense(1)(x[:, -1, :])
    model = Model(inputs=inp, outputs=x)
    model.compile(loss="mse", optimizer=Adam(learning_rate=p['learning_rate']))
    model.fit(X_seq, y_seq, batch_size=p['batch_size'], epochs=p['epochs'], verbose=0)

    # Construcción de secuencias para test
    X_full = np.vstack([X_train, X_test])
    X_pred_seq = []
    for i in range(len(training_set), len(training_set) + len(test_set)):
        X_pred_seq.append(X_full[i - p['window_size']:i])
    X_pred_seq = np.array(X_pred_seq)

    y_pred_scaled = model.predict(X_pred_seq)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()

    return model, pd.Series(y_pred, index=test_set.index, name='Transformer'), (scaler_x, scaler_y)


import optuna

def optimize_transformer_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            'window_size': trial.suggest_int('window_size', 6, 24),
            'head_size': trial.suggest_int('head_size', 16, 64),
            'num_heads': trial.suggest_int('num_heads', 1, 4),
            'ff_dim': trial.suggest_int('ff_dim', 32, 128),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),
            'epochs': trial.suggest_int('epochs', 20, 100),
            'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32]),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_transformer(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set['y'], y_pred)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape

            model_dict[trial.number] = {
                'params': model_params,
                'mape_best': best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)

    trials_data = []
    for trial in study.trials:
        trial_number = trial.number
        params = trial.params
        mape_mean = trial.value
        mape_best = model_dict.get(trial_number, {}).get('mape_best', np.inf)
        trials_data.append((trial_number, params, mape_mean, mape_best))

    trials_df = pd.DataFrame(
        trials_data,
        columns=['trial_number', 'params', 'mape_mean', 'mape_best']
    )

    return study.best_params, trials_df, model_dict

In [22]:
best_params_transformer, trials_df_transformer, model_dict_transformer = optimize_transformer_cv(df.assign(ds=df.index), split, n_trials=10)
trials_df_transformer['model'] = 'transformer'

[I 2025-05-04 01:33:03,144] A new study created in memory with name: no-name-9a2610ef-e355-4bf2-97ea-e7001b85ab36


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


[I 2025-05-04 01:33:33,336] Trial 0 finished with value: 0.3111893575094158 and parameters: {'window_size': 12, 'head_size': 64, 'num_heads': 2, 'ff_dim': 88, 'dropout': 0.1974706060076301, 'epochs': 89, 'batch_size': 16, 'learning_rate': 0.00026079240866928405}. Best is trial 0 with value: 0.3111893575094158.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


[I 2025-05-04 01:34:00,002] Trial 1 finished with value: 0.3772660818165471 and parameters: {'window_size': 20, 'head_size': 53, 'num_heads': 1, 'ff_dim': 46, 'dropout': 0.33356224973664594, 'epochs': 73, 'batch_size': 8, 'learning_rate': 0.00043933720723070177}. Best is trial 0 with value: 0.3111893575094158.


...

[I 2025-05-04 01:36:22,457] Trial 7 finished with value: 0.6312237401389065 and parameters: {'window_size': 14, 'head_size': 64, 'num_heads': 4, 'ff_dim': 74, 'dropout': 0.21293130201136667, 'epochs': 39, 'batch_size': 8, 'learning_rate': 0.00010280263856909105}. Best is trial 2 with value: 0.1790730761074292.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step


[I 2025-05-04 01:36:42,961] Trial 8 finished with value: 0.2704040289577092 and parameters: {'window_size': 10, 'head_size': 28, 'num_heads': 2, 'ff_dim': 42, 'dropout': 0.06043149157700034, 'epochs': 55, 'batch_size': 32, 'learning_rate': 0.001750967996271378}. Best is trial 2 with value: 0.1790730761074292.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


[I 2025-05-04 01:37:02,142] Trial 9 finished with value: 0.5161537909781281 and parameters: {'window_size': 15, 'head_size': 42, 'num_heads': 4, 'ff_dim': 72, 'dropout': 0.47097823768738706, 'epochs': 40, 'batch_size': 16, 'learning_rate': 0.0010696723323254593}. Best is trial 2 with value: 0.1790730761074292.


In [23]:
trials_df_transformer

,trial_number,params,mape_mean,mape_best,model
0,0,"{'window_size': 12, 'head_size': 64, 'num_head...",0.311189,0.192373,transformer
1,1,"{'window_size': 20, 'head_size': 53, 'num_head...",0.377266,0.138572,transformer
2,2,"{'window_size': 18, 'head_size': 48, 'num_head...",0.179073,0.109254,transformer
3,3,"{'window_size': 17, 'head_size': 61, 'num_head...",0.975806,0.251296,transformer
4,4,"{'window_size': 19, 'head_size': 42, 'num_head...",0.267186,0.085594,transformer
5,5,"{'window_size': 12, 'head_size': 62, 'num_head...",0.275173,0.101332,transformer
6,6,"{'window_size': 23, 'head_size': 20, 'num_head...",0.546981,0.128308,transformer
7,7,"{'window_size': 14, 'head_size': 64, 'num_head...",0.631224,0.111584,transformer
8,8,"{'window_size': 10, 'head_size': 28, 'num_head...",0.270404,0.126418,transformer
9,9,"{'window_size': 15, 'head_size': 42, 'num_head...",0.516154,0.185862,transformer


In [23]:
from lag_llama.gluon.estimator import LagLlamaEstimator
from lag_llama.gluon.lightning_module import LagLlamaLightningModule
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.modules.loss import NegativeLogLikelihood
from gluonts.torch.distributions.studentT import StudentTOutput
from gluonts.evaluation import make_evaluation_predictions
from sklearn.metrics import mean_absolute_percentage_error
import torch
import pandas as pd
import numpy as np
def fit_predict_eval_lagllama(training_set, test_set, model_params=None):
    context_length = model_params.get("context_length", 12)
    prediction_length = model_params.get("prediction_length", len(test_set))
    freq = "M"

    # Ordenar y limpiar
    training_set = training_set.sort_values("ds").reset_index(drop=True)
    test_set = test_set.sort_values("ds").reset_index(drop=True)

    # Dataset completo para forecast
    full_df = pd.concat([training_set, test_set]).reset_index(drop=True)

    full_ds = PandasDataset(
        [{"start": full_df['ds'].iloc[0], "target": full_df["y"].values}],
        freq=freq
    )

    estimator = LagLlamaEstimator(
        prediction_length=prediction_length,
        context_length=context_length,
        input_size=1,
        loss="nll",
        distr_output="student_t",
        trainer_kwargs=dict(max_epochs=model_params.get("epochs", 100))
    )

    predictor = estimator.train(full_ds)

    forecast_it, _ = make_evaluation_predictions(
        dataset=full_ds,
        predictor=predictor,
        num_samples=100
    )

    forecasts = list(forecast_it)
    yhat = forecasts[0].mean[-prediction_length:]

    return predictor, pd.Series(yhat, index=test_set.index, name="LagLlama"), None


import optuna

def optimize_lagllama_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            "context_length": trial.suggest_int("context_length", 6, 36),
            "prediction_length": trial.suggest_int("prediction_length", 6, 24),
            "epochs": trial.suggest_int("epochs", 10, 100)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index]
                test_set = df.iloc[test_index]

                _, y_pred, _ = fit_predict_eval_lagllama(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set["y"], y_pred)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape

            model_dict[trial.number] = {
                "params": model_params,
                "mape_best": best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame([
        {
            "trial_number": trial.number,
            "params": trial.params,
            "mape_mean": trial.value,
            "mape_best": model_dict.get(trial.number, {}).get("mape_best", np.inf)
        }
        for trial in study.trials
    ])

    return study.best_params, trials_df, model_dict

In [31]:
!git clone https://github.com/time-series-foundation-models/lag-llama/ 

Cloning into 'lag-llama'...
remote: Enumerating objects: 502, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 502 (delta 153), reused 109 (delta 109), pack-reused 312 (from 3)
Receiving objects: 100% (502/502), 282.40 KiB | 1.59 MiB/s, done.
Resolving deltas: 100% (251/251), done.


In [29]:
df_lagllama = df.reset_index()[["ds", "y"]].copy()
best_params_lagllama, trials_df_lagllama, model_dict_lagllama = optimize_lagllama_cv(df_lagllama, split, n_trials=10)
trials_df_lagllama["model"] = "lag-llama"

[I 2025-05-04 02:10:56,623] A new study created in memory with name: no-name-6b2d6a2d-ab14-4833-bc6e-2b690c34ccb2
[I 2025-05-04 02:10:56,627] Trial 0 finished with value: inf and parameters: {'context_length': 14, 'prediction_length': 17, 'epochs': 57}. Best is trial 0 with value: inf.
[I 2025-05-04 02:10:56,629] Trial 1 finished with value: inf and parameters: {'context_length': 21, 'prediction_length': 14, 'epochs': 43}. Best is trial 0 with value: inf.
[I 2025-05-04 02:10:56,634] Trial 2 finished with value: inf and parameters: {'context_length': 7, 'prediction_length': 24, 'epochs': 20}. Best is trial 0 with value: inf.
[I 2025-05-04 02:10:56,652] Trial 3 finished with value: inf and parameters: {'context_length': 6, 'prediction_length': 11, 'epochs': 71}. Best is trial 0 with value: inf.
[I 2025-05-04 02:10:56,666] Trial 4 finished with value: inf and parameters: {'context_length': 12, 'prediction_length': 16, 'epochs': 89}. Best is trial 0 with value: inf.
[I 2025-05-04 02:10:56,

Error en el ensayo con parámetros {'context_length': 14, 'prediction_length': 17, 'epochs': 57}: 1 validation error for LagLlamaEstimatorModel
loss
  value is not a valid dict (type=type_error.dict)
Error en el ensayo con parámetros {'context_length': 21, 'prediction_length': 14, 'epochs': 43}: 1 validation error for LagLlamaEstimatorModel
loss
  value is not a valid dict (type=type_error.dict)
Error en el ensayo con parámetros {'context_length': 7, 'prediction_length': 24, 'epochs': 20}: 1 validation error for LagLlamaEstimatorModel
loss
  value is not a valid dict (type=type_error.dict)
Error en el ensayo con parámetros {'context_length': 6, 'prediction_length': 11, 'epochs': 71}: 1 validation error for LagLlamaEstimatorModel
loss
  value is not a valid dict (type=type_error.dict)
Error en el ensayo con parámetros {'context_length': 12, 'prediction_length': 16, 'epochs': 89}: 1 validation error for LagLlamaEstimatorModel
loss
  value is not a valid dict (type=type_error.dict)
Error e

In [31]:
df_lagllama

,ds,y
0,2017-05-01,13534.398736
1,2017-06-01,11736.810614
2,2017-07-01,11922.816286
3,2017-08-01,12278.465143
4,2017-09-01,12063.154302
...,...,...
87,2024-08-01,13837.290250
88,2024-09-01,12648.582042
89,2024-10-01,16591.856305
90,2024-11-01,16015.728726
